In [1]:
# Centralized imports (cleaned)
# from bioio import BioImage
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tifffile import imwrite, imread
import tifffile
from skimage.segmentation import expand_labels, clear_border
from skimage.measure import regionprops_table
from cellpose import models
import napari
from liffile import LifFile
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import colorsys
from scipy.spatial import KDTree
from skimage.measure import regionprops
from skimage.transform import resize, rescale
from skimage.morphology import remove_small_objects
from pathlib import Path
import pandas as pd
import scipy.ndimage as ndi
from skimage.measure import regionprops
# from instanseg import InstanSeg  # InstanSeg disabled: focusing on cellpose 3D vs original 2D
import time
from cellpose import models, utils as cellpose_utils
# from micro_sam.automatic_segmentation import get_predictor_and_segmenter, automatic_3d_segmentation

available = torch.cuda.is_available()
device_count = torch.cuda.device_count() if available else 0
device_name = torch.cuda.get_device_name(0) if available and device_count > 0 else None

status = {
    "cuda_available": available,
    "device_count": device_count,
    "device_name": device_name,
}
print(status)

{'cuda_available': True, 'device_count': 1, 'device_name': 'NVIDIA GeForce RTX 3080'}


In [2]:

_MODEL_CACHE = {}  # cache loaded models across instances so we don't reload them per image

class SegmentationComparisons:
    """Parameter-scan comparison of nuclear-segmentation methods on a DAPI z-stack.

    The input image is a 2-channel z-stack stored as (Z, C, Y, X); only the
    DAPI channel (`dapi_channel`) is used. The downsampled DAPI is Gaussian-smoothed
    (preprocessing) and that smoothed volume is fed to every method.

    Methods:
      - `original`:        custom cellpose model, per-z 2D then stitched in 3D.
                           This is the FIXED BASELINE (NOT scanned), computed once and
                           reused as the reference in every comparison.
      - `cellpose_true3d`: custom cellpose model, native 3D. Scanned over
                           cellprob_threshold / flow_threshold; `min_size` is applied
                           afterwards as cheap post-processing (no re-segmentation).
      - `instanseg`:       DISABLED (kept commented out for reference).

    Per image we save one large comparison PNG (top row: original image + each 3D seg;
    bottom row: original 2D-stitched seg + each 3D-vs-original diff) and one CSV per
    parameter set in its own subfolder of `output_dir`.
    """

    def __init__(self, input_csv=None, index=None, image_path=None, image_name=None,
        scale_factor_xy=3, scale_factor_z=2,
        custom_model_path=r"Z:\Bel\Jorge_SPACEFISH_Examples\v115-2ch\spacefish_custom_tissue",
        output_dir=Path(r"Z:\Bel\Jorge_SPACEFISH_Examples\cellpose_comparisons\round_2_comparisons"),
        dapi_channel=0, dapi_sigma_xy=1.0, dapi_sigma_z=0.5):
        """Provide the input image EITHER via a CSV (`input_csv` + `index`) OR via a
        direct file path (`image_path`). Passing `image_path` ignores the CSV entirely,
        so you can iterate over a folder of images without building a spreadsheet."""
        self.input_csv = input_csv
        self.index = index
        self.scale_factor_xy = float(scale_factor_xy)
        self.scale_factor_z = float(scale_factor_z)
        self.custom_model_path = custom_model_path

        # Gaussian smoothing (preprocessing) applied to the downsampled DAPI
        self.dapi_sigma_xy = float(dapi_sigma_xy)
        self.dapi_sigma_z = float(dapi_sigma_z)
        self.dapi_channel = int(dapi_channel)

        if image_path is not None:
            # ---- folder / direct-path mode ----
            self.two_channel_image_path = Path(image_path)
            self.image_name = image_name if image_name is not None else self.two_channel_image_path.stem
        elif input_csv is not None and index is not None:
            # ---- CSV mode (original behaviour) ----
            row = input_csv.loc[index]
            self.two_channel_image_path = Path(row["2_channel_tif_save_path"])
            self.image_name = row["image_name"]
        else:
            raise ValueError("Provide either `image_path`, or both `input_csv` and `index`.")

        self.output_dir = Path(output_dir)

        # (Z, C, Y, X)
        self.two_channel_image = tifffile.imread(self.two_channel_image_path)
        if self.two_channel_image.ndim != 4:
            raise ValueError(f"Expected a 4D (Z, C, Y, X) image, got shape {self.two_channel_image.shape}")

        self.results = {}  # method -> label array
        self.counts = {}   # method -> object count

    def pixel_size(self):
        with tifffile.TiffFile(self.two_channel_image_path) as tif:
            tags = {tag.name: tag.value for tag in tif.pages[0].tags.values()}
            x_um = 1 / (tags["XResolution"][0] / tags["XResolution"][1])
            y_um = 1 / (tags["YResolution"][0] / tags["YResolution"][1])
            try:
                z_um = float(str(tags["IJMetadata"]).split("nscales=")[1].split(",")[2].split("\\nunit")[0])
            except Exception:
                z_um = float(str(tags["ImageDescription"]).split("spacing=")[1].split("loop")[0])
        self.original_spacing = (x_um, y_um, z_um)

    @staticmethod
    def filtering(img: np.ndarray, sigma_xy: float, sigma_z: float) -> np.ndarray:
        """Applies Gaussian filtering to the image.
        Args:
            img (np.ndarray): Image array to be filtered.
            sigma_xy (float): Sigma value for Gaussian filter in xy dimensions.
            sigma_z (float): Sigma value for Gaussian filter in z dimension.
        Returns:
            np.ndarray: Filtered image array.
        """
        sigma = [sigma_z, sigma_xy, sigma_xy] if img.ndim == 3 else [sigma_xy, sigma_xy]
        radius = [int(sigma_z), int(sigma_xy), int(sigma_xy)] if img.ndim == 3 else [int(sigma_xy), int(sigma_xy)]
        return ndi.gaussian_filter(input=img, sigma=sigma, radius=radius)

    def rescale_image(self):
        """Downsample the DAPI channel, Gaussian-smooth it, and compute z-anisotropy."""
        self.pixel_size()
        x_um, y_um, z_um = self.original_spacing
        xy_ratio = 1.0 / self.scale_factor_xy
        z_ratio = 1.0 / self.scale_factor_z

        dapi = self.two_channel_image[:, self.dapi_channel].astype(np.float32)  # (Z, Y, X)
        self.dapi_ds = rescale(dapi, (z_ratio, xy_ratio, xy_ratio),
                               anti_aliasing=True, preserve_range=True).astype(np.float32)

        # preprocessing: Gaussian smoothing applied to ALL methods (incl. the baseline)
        self.dapi_ds = self.filtering(self.dapi_ds, self.dapi_sigma_xy, self.dapi_sigma_z).astype(np.float32)

        # physical voxel spacing after downsampling
        self.z_spacing_ds = z_um * self.scale_factor_z
        self.xy_spacing_ds = x_um * self.scale_factor_xy
        self.anisotropy = self.z_spacing_ds / self.xy_spacing_ds
        self.pixel_size_um_2d = float(self.xy_spacing_ds)
        print(f"downsampled+smoothed DAPI shape {self.dapi_ds.shape}, anisotropy {self.anisotropy:.3f} "
              f"(sigma_xy={self.dapi_sigma_xy}, sigma_z={self.dapi_sigma_z})")

    @staticmethod
    def _get_cellpose_model(pretrained=None):
        key = str(pretrained)
        if key not in _MODEL_CACHE:
            if pretrained is None:
                _MODEL_CACHE[key] = models.CellposeModel(gpu=True)  # built-in CPSAM
            else:
                _MODEL_CACHE[key] = models.CellposeModel(gpu=True, pretrained_model=str(pretrained))
        return _MODEL_CACHE[key]

    @staticmethod
    def _label_count(seg):
        return int(np.unique(seg).size - (1 if (seg == 0).any() else 0))

    @staticmethod
    def _apply_min_size(seg, min_size):
        """Post-process a label volume by removing objects smaller than `min_size`
        voxels. Cheap alternative to re-running segmentation for every min_size."""
        if min_size and int(min_size) > 0:
            return remove_small_objects(seg.astype(np.int32), min_size=int(min_size))
        return seg

    # ---- segmentation methods: each RETURNS a label volume ----

    def cellpose_custom_2d_stitched(self, stitch_threshold=0.1, cellprob_threshold=0.0, min_size=15):
        """BASELINE 'original': custom cellpose model on DAPI, per-z 2D then stitched in 3D."""
        model = self._get_cellpose_model(self.custom_model_path)
        seg, _, _ = model.eval(self.dapi_ds, z_axis=0, do_3D=False,
                               stitch_threshold=stitch_threshold, anisotropy=self.anisotropy,
                               cellprob_threshold=cellprob_threshold, min_size=min_size)
        return seg

    def cellpose_custom_3d(self, cellprob_threshold=0.0, flow_threshold=0.4, min_size=15):
        """Custom cellpose model on DAPI, native 3D (stitch_threshold is unused in do_3D mode)."""
        model = self._get_cellpose_model(self.custom_model_path)
        seg, _, _ = model.eval(self.dapi_ds, z_axis=0, do_3D=True, anisotropy=self.anisotropy,
                               cellprob_threshold=cellprob_threshold, flow_threshold=flow_threshold,
                               min_size=min_size)
        return seg

    # ---- InstanSeg DISABLED (kept for reference) ----
    # def instanseg_2d_stitched(self, stitch_threshold=0.1, pixel_size_scale=1.0, target="nuclei"):
    #     """InstanSeg per-z (DAPI only) then stitched in 3D.
    #
    #     - `stitch_threshold`: IoU cutoff for linking 2D masks across z.
    #     - `pixel_size_scale`: multiplies the pixel size handed to InstanSeg (object-scale prior).
    #     - `target`: which InstanSeg output to keep ('nuclei' or 'cells').
    #     """
    #     if "instanseg" not in _MODEL_CACHE:
    #         _MODEL_CACHE["instanseg"] = InstanSeg("fluorescence_nuclei_and_cells", verbosity=0)
    #     model = _MODEL_CACHE["instanseg"]
    #     pixel_size = self.pixel_size_um_2d * float(pixel_size_scale)
    #     z_masks = []
    #     for z in range(self.dapi_ds.shape[0]):
    #         plane = self.dapi_ds[z][None]  # (C=1, H, W), DAPI only
    #         labeled, _ = model.eval_small_image(plane, pixel_size, target=target)
    #         lab = np.asarray(labeled.cpu() if hasattr(labeled, "cpu") else labeled).squeeze()
    #         if lab.ndim == 3:  # (n_outputs, H, W) -> take the first output
    #             lab = lab[0]
    #         z_masks.append(lab.astype(np.uint32))
    #     stacked = np.stack(z_masks, axis=0)
    #     return cellpose_utils.stitch3D(stacked, stitch_threshold=stitch_threshold)

    # ---- output helpers ----

    @staticmethod
    def _align(arr, ref_shape):
        """Nearest-neighbour resize to ref_shape if shapes differ (handles +/-1px rounding)."""
        if arr.shape == ref_shape:
            return arr
        return resize(arr, ref_shape, order=0, anti_aliasing=False,
                      preserve_range=True).astype(arr.dtype)

    def _save_grid_png(self, dapi_mip, original_mip, original_count,
                       seg_mips, seg_counts, seg_labels, png_path, max_variants_per_row=4):
        """Wrapped comparison figure for many 3D segmentations.

        Each block uses 2 rows:
          top row:    original image (DAPI MIP) | each cellpose-3D segmentation overlay
          bottom row: original 2D-stitched seg  | diff of each 3D seg vs the original 2D

        If there are many 3D parameter sets, they are wrapped across multiple blocks so
        the PNG stays readable instead of becoming one extremely wide strip.
        """
        n = len(seg_mips)
        variants_per_row = max(1, int(max_variants_per_row))
        nblocks = int(np.ceil(n / variants_per_row)) if n > 0 else 1
        ncols = variants_per_row + 1

        base_norm = dapi_mip.astype(np.float32)
        base_norm = (base_norm - base_norm.min()) / (base_norm.max() - base_norm.min() + 1e-8)

        def overlay(ax, labels, title):
            ax.imshow(dapi_mip, cmap="gray")
            if labels is not None:
                masked = np.ma.masked_where(labels == 0, labels)
                ax.imshow(masked, cmap="nipy_spectral", alpha=0.5, interpolation="nearest")
            ax.set_title(title)

        def diff(ax, other, title):
            rgb = np.stack([base_norm, base_norm, base_norm], axis=-1)
            if original_mip is not None and other is not None:
                orig_b = original_mip > 0
                other_b = other > 0
                rgb[other_b & ~orig_b] = [0.0, 1.0, 0.0]   # green: in 3D but not original
                rgb[orig_b & ~other_b] = [1.0, 0.0, 0.0]   # red: in original but not 3D
            ax.imshow(rgb, interpolation="nearest")
            ax.set_title(title)

        fig, axes = plt.subplots(2 * nblocks, ncols, figsize=(6 * ncols, 6 * nblocks), squeeze=False)

        for block in range(nblocks):
            start = block * variants_per_row
            end = min(start + variants_per_row, n)
            top_row = 2 * block
            bottom_row = top_row + 1

            axes[top_row, 0].imshow(dapi_mip, cmap="gray")
            axes[top_row, 0].set_title("Original image (DAPI MIP)")
            overlay(axes[bottom_row, 0], original_mip,
                    f"Original seg (cellpose 2D stitched)\nn={original_count}")

            for col_offset, idx in enumerate(range(start, end), start=1):
                overlay(axes[top_row, col_offset], seg_mips[idx],
                        f"{seg_labels[idx]}\nn={seg_counts[idx]}")
                diff(axes[bottom_row, col_offset], seg_mips[idx],
                     f"Diff vs original 2D\n{seg_labels[idx]}\n(green=3D only, red=orig only)")

            for col in range(end - start + 1, ncols):
                axes[top_row, col].axis("off")
                axes[bottom_row, col].axis("off")

        for ax in axes.ravel():
            if ax.axison:
                ax.axis("off")
        fig.suptitle(self.image_name)
        fig.tight_layout()
        fig.savefig(png_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

    ######### MAIN PART ############
    def run(self,
            # fixed baseline (original): cellpose 2D + stitch, NOT scanned
            original_stitch_threshold=0.1,
            original_cellprob_threshold=0.0,
            original_min_size=15,
            # cellpose true-3D scan grid
            cellpose3d_cellprob_thresholds=(-2.0, 0.0, 2.0),
            cellpose3d_flow_thresholds=(0.4,),
            cellpose3d_min_sizes=(15,),
            # # instanseg scan grid (DISABLED)
            # instanseg_stitch_thresholds=(0.05, 0.1, 0.2),
            # instanseg_pixel_size_scales=(1.0,),
            # instanseg_target="nuclei",
            save_tifs=False):
        """Compute the fixed `original` baseline once, then scan cellpose true-3D.

        Optimisation: the 3D segmentation is run ONCE per unique
        (cellprob_threshold, flow_threshold) pair, and each `min_size` is applied
        afterwards as a cheap post-processing filter (no re-segmentation).

        Outputs per image: one large comparison PNG (top row = original image + each
        3D seg; bottom row = original 2D-stitched seg + each 3D-vs-original diff) plus
        one results row per parameter set. TIF code is kept but disabled (save_tifs=False).
        """
        self.rescale_image()
        self.output_dir.mkdir(parents=True, exist_ok=True)

        # --- baseline 'original' computed ONCE, reused for every comparison ---
        t0 = time.time()
        original_seg = self.cellpose_custom_2d_stitched(
            stitch_threshold=original_stitch_threshold,
            cellprob_threshold=original_cellprob_threshold,
            min_size=original_min_size)
        original_time = time.time() - t0
        original_count = self._label_count(original_seg)
        dapi_mip = self.dapi_ds.max(axis=0)
        ref_shape = dapi_mip.shape
        original_mip = self._align(
            original_seg.max(axis=0) if original_seg.ndim == 3 else original_seg, ref_shape)
        print(f"  original (baseline): {original_count} objects, {original_time:.1f}s")

        # --- run cellpose 3D ONCE per unique (cellprob, flow); min_size is post-processing ---
        base_segs = {}       # (cp, fl) -> raw label volume (no size filter)
        base_seg_times = {}  # (cp, fl) -> seconds
        for cp in cellpose3d_cellprob_thresholds:
            for fl in cellpose3d_flow_thresholds:
                t0 = time.time()
                base_segs[(cp, fl)] = self.cellpose_custom_3d(
                    cellprob_threshold=cp, flow_threshold=fl, min_size=0)
                base_seg_times[(cp, fl)] = time.time() - t0
                print(f"  cellpose3d seg cellprob={cp} flow={fl}: {base_seg_times[(cp, fl)]:.1f}s")

        # --- apply each min_size as cheap post-processing on the cached segmentations ---
        run_results = []
        seg_mips, seg_counts, seg_labels = [], [], []
        # record the baseline once per image (param_set == "original")
        run_results.append({"image_name": self.image_name, "param_set": "original",
                            "method": "original", "time_taken": original_time,
                            "objects_found": original_count,
                            "stitch_threshold": original_stitch_threshold,
                            "cellprob_threshold": original_cellprob_threshold,
                            "min_size": original_min_size})

        for cp in cellpose3d_cellprob_thresholds:
            for fl in cellpose3d_flow_thresholds:
                raw = base_segs[(cp, fl)]
                for ms in cellpose3d_min_sizes:
                    t0 = time.time()
                    seg = self._apply_min_size(raw, ms)
                    post_time = time.time() - t0
                    total_time = base_seg_times[(cp, fl)] + post_time
                    count = self._label_count(seg)
                    folder = f"cellpose_true3d__cellprob_{cp}__flow_{fl}__minsize_{ms}"
                    label = f"cp={cp}, fl={fl}, min={ms}"
                    print(f"  {folder}: {count} objects ({post_time:.2f}s post)")

                    # TIF code kept but disabled for this run (save_tifs=False)
                    if save_tifs:
                        set_dir = self.output_dir / folder
                        set_dir.mkdir(parents=True, exist_ok=True)
                        imwrite(set_dir / f"{self.image_name}__cellpose_true3d.tif",
                                seg.astype(np.uint32))

                    seg_mips.append(self._align(seg.max(axis=0) if seg.ndim == 3 else seg, ref_shape))
                    seg_counts.append(count)
                    seg_labels.append(label)

                    run_results.append({"image_name": self.image_name, "param_set": folder,
                                        "method": "cellpose_true3d", "time_taken": total_time,
                                        "objects_found": count, "cellprob_threshold": cp,
                                        "flow_threshold": fl, "min_size": ms})

        # --- one large comparison figure per image ---
        self._save_grid_png(dapi_mip, original_mip, original_count,
                            seg_mips, seg_counts, seg_labels,
                            self.output_dir / f"{self.image_name}__cellpose3d_comparison.png")

        return run_results


In [5]:
# input_csv = pd.read_excel(r"z:\Bel\Jorge_SPACEFISH_Examples\image_locations.xlsx")
# input_csv.head()


In [ ]:
# Parameter-scan comparison on every image.
# Original (cellpose 2D + stitch) is the FIXED baseline; only cellpose true-3D is scanned
# (InstanSeg disabled). This run saves PNGs only (save_tifs=False).
#
# INPUT MODE: iterate over all images in a folder (no CSV needed this time).
# To revert to the CSV, set USE_CSV = True and the loop below uses input_csv/index instead.
USE_CSV = False
input_csv = None  # set to a DataFrame again if you switch USE_CSV back to True

input_folder = Path(r"E:\two_channel_images")
output_root = Path(r"E:\cellpose_outputs")
image_extensions = (".tif", ".tiff")

if USE_CSV:
    if input_csv is None:
        raise ValueError("Set input_csv before using USE_CSV=True.")
    inputs = [{"input_csv": input_csv, "index": idx} for idx in input_csv.index]
else:
    image_paths = sorted(p for p in input_folder.iterdir()
                         if p.is_file() and p.suffix.lower() in image_extensions)
    inputs = [{"image_path": p} for p in image_paths[4:]]
    print(f"found {len(inputs)} images in {input_folder}")

all_results = []
for kw in inputs:
    # output_dir is shared; the combined comparison PNG is saved at output_root, and a
    # results.csv is written per parameter-set subfolder.
    comparison = SegmentationComparisons(scale_factor_xy=3, scale_factor_z=2,
                                         output_dir=output_root, **kw)
    image_results = comparison.run(
        # cellpose true-3D grid: segmentation runs ONCE per (cellprob, flow);
        # min_size is applied afterwards as cheap post-processing.
        cellpose3d_cellprob_thresholds=(-2.0, -1, 0.0, 2.0),
        cellpose3d_flow_thresholds=(0.2, 0.4, 0.6),
        cellpose3d_min_sizes=(15,),
        # instanseg disabled -- focusing on cellpose 3D vs original 2D stitched
        # instanseg_stitch_thresholds=(0.05, 0.1, 0.2),
        # instanseg_pixel_size_scales=(1.0,),
        save_tifs=False,
    )
    all_results.extend(image_results)

results_df = pd.DataFrame(all_results)

# one CSV per parameter-set folder (accumulated across all images)
for param_set, group in results_df.groupby("param_set"):
    if param_set == "original":
        out_path = output_root / "results_original.csv"
    else:
        out_path = output_root / param_set / "results.csv"
        out_path.parent.mkdir(parents=True, exist_ok=True)
    group.to_csv(out_path, index=False)
    print(f"saved {len(group)} rows -> {out_path}")

# single combined CSV across all images and parameter sets
results_csv_path = output_root / "segmentation_comparison_results.csv"
results_df.to_csv(results_csv_path, index=False)
print(f"saved {len(results_df)} rows -> {results_csv_path}")
results_df


found 36 images in E:\two_channel_images
downsampled+smoothed DAPI shape (63, 1303, 1311), anisotropy 2.813 (sigma_xy=1.0, sigma_z=0.5)


100%|██████████| 62/62 [00:01<00:00, 38.25it/s]


  original (baseline): 302 objects, 519.4s
  cellpose3d seg cellprob=-2.0 flow=0.2: 1250.9s
  cellpose3d seg cellprob=-2.0 flow=0.4: 1246.7s
  cellpose3d seg cellprob=-2.0 flow=0.6: 1245.5s
  cellpose3d seg cellprob=-1 flow=0.2: 1227.9s
  cellpose3d seg cellprob=-1 flow=0.4: 1230.5s
  cellpose3d seg cellprob=-1 flow=0.6: 1231.8s
  cellpose3d seg cellprob=0.0 flow=0.2: 1235.5s
  cellpose3d seg cellprob=0.0 flow=0.4: 1232.5s
  cellpose3d seg cellprob=0.0 flow=0.6: 1232.8s
  cellpose3d seg cellprob=2.0 flow=0.2: 1230.9s
  cellpose3d seg cellprob=2.0 flow=0.4: 1232.8s
  cellpose3d seg cellprob=2.0 flow=0.6: 1233.0s
  cellpose_true3d__cellprob_-2.0__flow_0.2__minsize_15: 275 objects (0.78s post)
  cellpose_true3d__cellprob_-2.0__flow_0.4__minsize_15: 275 objects (0.83s post)
  cellpose_true3d__cellprob_-2.0__flow_0.6__minsize_15: 275 objects (0.78s post)
  cellpose_true3d__cellprob_-1__flow_0.2__minsize_15: 273 objects (0.81s post)
  cellpose_true3d__cellprob_-1__flow_0.4__minsize_15: 273 o

no seeds found in get_masks_torch - no masks found.
100%|██████████| 47/47 [00:00<00:00, 118.29it/s]


  original (baseline): 35 objects, 37.5s
  cellpose3d seg cellprob=-2.0 flow=0.2: 339.9s
  cellpose3d seg cellprob=-2.0 flow=0.4: 339.5s
  cellpose3d seg cellprob=-2.0 flow=0.6: 340.4s
  cellpose3d seg cellprob=-1 flow=0.2: 339.5s
  cellpose3d seg cellprob=-1 flow=0.4: 339.1s
  cellpose3d seg cellprob=-1 flow=0.6: 339.4s
  cellpose3d seg cellprob=0.0 flow=0.2: 338.2s
  cellpose3d seg cellprob=0.0 flow=0.4: 339.5s
  cellpose3d seg cellprob=0.0 flow=0.6: 338.7s
  cellpose3d seg cellprob=2.0 flow=0.2: 338.9s
  cellpose3d seg cellprob=2.0 flow=0.4: 338.7s
  cellpose3d seg cellprob=2.0 flow=0.6: 338.8s
  cellpose_true3d__cellprob_-2.0__flow_0.2__minsize_15: 32 objects (0.16s post)
  cellpose_true3d__cellprob_-2.0__flow_0.4__minsize_15: 32 objects (0.16s post)
  cellpose_true3d__cellprob_-2.0__flow_0.6__minsize_15: 32 objects (0.16s post)
  cellpose_true3d__cellprob_-1__flow_0.2__minsize_15: 32 objects (0.16s post)
  cellpose_true3d__cellprob_-1__flow_0.4__minsize_15: 32 objects (0.16s post)

100%|██████████| 47/47 [00:02<00:00, 20.28it/s]


  original (baseline): 284 objects, 167.3s
  cellpose3d seg cellprob=-2.0 flow=0.2: 1650.0s
  cellpose3d seg cellprob=-2.0 flow=0.4: 1649.2s
  cellpose3d seg cellprob=-2.0 flow=0.6: 1650.5s
  cellpose3d seg cellprob=-1 flow=0.2: 1650.5s
  cellpose3d seg cellprob=-1 flow=0.4: 1649.5s
  cellpose3d seg cellprob=-1 flow=0.6: 1649.4s
  cellpose3d seg cellprob=0.0 flow=0.2: 1648.4s
  cellpose3d seg cellprob=0.0 flow=0.4: 1650.0s
  cellpose3d seg cellprob=0.0 flow=0.6: 1649.2s
